In [8]:
import convexity_of_space
from lmc.models.models_mlp import MLP, WMLP

tmp_wmlp, [wmlp1, wmlp2] = convexity_of_space.sample_models(
    num_models=2, model_class=WMLP, model_args=convexity_of_space.default_wmlp_hyperparams())
tmp_mlp, [mlp1, mlp2] = convexity_of_space.sample_models(
    num_models=2, model_class=MLP, model_args=convexity_of_space.default_mlp_hyperparams())

mean_losses_wmlp = convexity_of_space.compute_losses_on_line(wmlp1, wmlp2, tmp_wmlp, num_steps=100)
mean_losses_mlp = convexity_of_space.compute_losses_on_line(mlp1, mlp2, tmp_mlp, num_steps=100)

100it [06:14,  3.74s/it]
100it [06:13,  3.73s/it]


In [2]:
convexity_of_space.compute_losses_on_line(mlp1, mlp2, tmp_mlp, num_steps=3)

3it [00:26,  8.83s/it]


[2.303105115890503, 2.3033437728881836, 2.3068430423736572]

In [36]:
import pickle

runs_data = []
for i in range(71):
    with open(f"mlp-wmlp-{i}.pickle", "rb") as file:
        runs_data.append(pickle.load(file))
runs_data = runs_data[30:]

In [7]:
runs_data[0].keys()

dict_keys(['wmlp1', 'wmlp2', 'mlp1', 'mlp2', 'mean_losses_wmlp', 'mean_losses_mlp'])

In [35]:
import plotly.express as px
import plotly.colors
import numpy as np
import pandas as pd

#mean_losses_mlp, mean_losses_wmlp = (r := runs_data[4])["mean_losses_mlp"], r["mean_losses_wmlp"]

px.line(pd.DataFrame(
    [dict(x=x, y=(mean_loss-np.mean(mean_losses))/np.std(mean_losses), run=run_type+f"-{i}")
        for i, run in enumerate(runs_data)
        for run_type, mean_losses in [("WMLP", run["mean_losses_wmlp"]), ("MLP", run["mean_losses_mlp"])]
        for mean_loss, x in zip(mean_losses, np.linspace(0, 1, len(mean_losses)))
        if run_type == "WMLP"
    ]),
    x="x", y="y", color="run",
    title="Loss on part MNIST dataset of 100 networks linearly interpolated between random untrained networks\n",
    subtitle="(zero-centered and scaled by respective stddev/curves not to scale!)",
    height=1000,
    color_discrete_map={**{f"WMLP-{i}": plotly.colors.DEFAULT_PLOTLY_COLORS[0] for i in range(len(runs_data))},
                        **{f"MLP-{i}": plotly.colors.DEFAULT_PLOTLY_COLORS[1] for i in range(len(runs_data))}}
)

In [38]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import torch
import sklearn.decomposition
import itertools

def parameters_to_vector(model):
    return torch.concat([param_tensor.reshape(-1) for param_tensor in model.parameters()])

def plotly_draw_line(fig, x_from, y_from, x_to, y_to, color):
    fig.add_trace(go.Scatter(
        x=[x_from, x_to],
        y=[y_from, y_to],
        mode='lines',
        line=dict(color=color, width=2),
        showlegend=False
    ))

def soft_extents(array, index):
    array = sorted(array)
    return array[index], array[-index]

param_matrix = np.stack([
    parameters_to_vector(model).detach().cpu().numpy().astype(np.float64)
    for run in runs_data for model in [run["wmlp1"], run["mlp1"], run["wmlp2"], run["mlp2"]]
], axis=0)
param_pca = sklearn.decomposition.PCA(2, svd_solver="full").fit_transform(param_matrix)
mean_losses_matrix = np.stack([
    mean_losses
    for run in runs_data for mean_losses in [run["mean_losses_wmlp"], run["mean_losses_mlp"]]
], axis=0)

mlp_mode = 1

# param_pca: [4*run_id: wmlp1, 4*run_id+1: mlp1, 4*run_id+2: wmlp2, 4*run_id+3: mlp2]
# mean_losses_matrix: [2*run_id: wmlp (between 1 and 2), 2*run_id+1: mlp (between 1 and 2)]
fig = go.Figure()
all_convexity_errors = []
for run_id in range(len(runs_data)):
    mean_losses = mean_losses_matrix[2*run_id+mlp_mode]

    def convexity_error(x, y, x1, y1, x2, y2):
        if not x1 <= x < x2:
            return 0.
        # if y > y1 + (y2-y1)/(x2-x1)*(x-x1):
        return (y - (y1 + (y2-y1)/(x2-x1)*(x-x1))) # / (max(y, y1, y2) - min(y, y1, y2))
        # return 0.

    loss_points = np.stack([np.linspace(0, 1, len(mean_losses)), mean_losses], axis=1)
    convexity_errors = [
        (max(
            convexity_error(*p, *p1, *p2)
            for p1 in loss_points[:i]
            for p2 in loss_points[i+1:]
        ) / (max(mean_losses) - min(mean_losses)))
        if i > 0 and i < len(loss_points) - 1 else 0.
        for i, p in enumerate(loss_points)
    ]
    all_convexity_errors.append(convexity_errors)

    param_from, param_to = param_pca[4*run_id+mlp_mode], param_pca[4*run_id+2+mlp_mode]
    param_line_points = np.linspace(param_from, param_to, len(mean_losses))
    for i, error in enumerate(convexity_errors[:-1]):
        # color = px.colors.sample_colorscale('haline', error)[0]
        color = px.colors.sample_colorscale('PuOr', 1-(error  * 0.5 + 0.5))[0]
        plotly_draw_line(fig, *param_line_points[i], *param_line_points[i+1], color)

fig.update_layout(title="", xaxis_title="X", yaxis_title="Y", width=1000, height=1000,
                   xaxis_range=soft_extents(param_pca[mlp_mode::2,0], 3), yaxis_range=soft_extents(param_pca[mlp_mode::2,1], 3),

    plot_bgcolor='#272822',
        xaxis=dict(color='white', gridcolor='#373832', zeroline=False),
        yaxis=dict(color='white', gridcolor='#373832', zeroline=False),
    )
fig.show()


In [13]:
for run_id, convexity_errors in enumerate(all_convexity_errors):
    print(f"Run {run_id}: Worst convexity error = {max(convexity_errors)}")
(np.array(all_convexity_errors).max(axis=1) < 1e-7).mean()

Run 0: Worst convexity error = 0.0
Run 1: Worst convexity error = 0.0
Run 2: Worst convexity error = 0.0
Run 3: Worst convexity error = 0.0
Run 4: Worst convexity error = 0.0
Run 5: Worst convexity error = 0.0
Run 6: Worst convexity error = 0.0
Run 7: Worst convexity error = 0.0
Run 8: Worst convexity error = 0.0
Run 9: Worst convexity error = 0.0
Run 10: Worst convexity error = 0.0
Run 11: Worst convexity error = 0.0
Run 12: Worst convexity error = 0.0
Run 13: Worst convexity error = 0.0
Run 14: Worst convexity error = 0.0
Run 15: Worst convexity error = 0.0
Run 16: Worst convexity error = 0.0
Run 17: Worst convexity error = 0.0
Run 18: Worst convexity error = 0.0
Run 19: Worst convexity error = 0.0
Run 20: Worst convexity error = 0.0
Run 21: Worst convexity error = 0.0
Run 22: Worst convexity error = 0.0
Run 23: Worst convexity error = 0.0
Run 24: Worst convexity error = 0.0
Run 25: Worst convexity error = 0.0
Run 26: Worst convexity error = 0.0
Run 27: Worst convexity error = 0.0
Ru

np.float64(1.0)

In [33]:
import plotly.express as px
import plotly.colors
import pandas as pd

fig = px.line(pd.DataFrame(
    [dict(x=x, y=(mean_loss-np.min(mean_losses))/((max(mean_losses) - min(mean_losses)) or 1)
          + 1.15 * (i + (run_type == "MLP") * (2 + len(runs_data))), run=run_type+f"-{i}")
        for i, run in enumerate(runs_data)
        for run_type, mean_losses in [("WMLP", run["mean_losses_wmlp"]), ("MLP", run["mean_losses_mlp"])]
        for mean_loss, x in zip(mean_losses, np.linspace(0, 1, len(mean_losses)))
    ]),
    x="x", y="y", color="run",
    title="Loss on part MNIST dataset of 100 networks linearly interpolated between random untrained networks\n",
    subtitle="(zero-centered and scaled by respective stddev/curves not to scale!)",
    width=500,
    height=5000,
    color_discrete_map={**{f"WMLP-{i}": plotly.colors.DEFAULT_PLOTLY_COLORS[0] for i in range(len(runs_data))},
                        **{f"MLP-{i}": plotly.colors.DEFAULT_PLOTLY_COLORS[1] for i in range(len(runs_data))}},
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0), showlegend=False)
fig